Test WORKING Part

BAsh, to link C++

c++ -O3 -Wall -std=c++17 -shared -undefined dynamic_lookup \
$(python3 -m pybind11 --includes) \
library.cpp \
-o library$(python3-config --extension-suffix)

In [ ]:
c++ -O3 -Wall -std=c++17 -shared -undefined dynamic_lookup \
$(python3 -m pybind11 --includes) \
PDE_FD.cpp \
-o USoptions_lib$(python3-config --extension-suffix)

In [ ]:
rm -rf build/  # remove old build
python setup.py build_ext --inplace
python3 setup_us_options_li.py build_ext --inplace

In [33]:
underlying = "AAPL"
expiry = "2026-03-20"
current_price = 265

df_orderbook = get_option_orderbook_view(underlying, expiry, current_price, half=10, step_size=5)
print(df_orderbook)


    call_bid_qty  call_bid_price  call_theo  call_ask_price  call_ask_qty  \
0           20.0           50.00      50.68           51.12          19.0   
1           39.0           44.35      45.75           46.71           5.0   
2           15.0           39.79      40.86           40.99          15.0   
3           19.0           36.13      36.03           36.43          19.0   
4            5.0           31.36      31.28           32.03           7.0   
5           59.0           26.72      26.66           27.12           5.0   
6           39.0           22.35      22.23           23.02          75.0   
7           15.0           17.80      18.05           18.36          20.0   
8           19.0           14.26      14.20           14.51           2.0   
9           58.0           10.34      10.77           10.66          20.0   
10          20.0            7.62       7.83            7.66           1.0   
11          67.0            4.95       5.41            5.05          60.0   

In [27]:
from datetime import date, timedelta
import calendar

def get_available_expiries(symbol, months_ahead=12):
    """Return a list of standard NYSE option expiries (3rd Friday) for next N months."""
    expiries = []
    today = date.today()

    for m in range(months_ahead):
        # Calculate target month and year
        month = (today.month + m - 1) % 12 + 1
        year = today.year + ((today.month + m - 1) // 12)

        # Find third Friday
        c = calendar.Calendar(firstweekday=calendar.MONDAY)
        month_days = [day for day in c.itermonthdates(year, month) if day.month == month]
        fridays = [d for d in month_days if d.weekday() == 4]  # Friday=4
        third_friday = fridays[2]  # third Friday
        expiries.append(third_friday.strftime("%Y-%m-%d"))

    return expiries

# Example:
print(get_available_expiries("AAPL"))
# ['2026-03-20', '2026-04-17', '2026-05-15', ...]

['2026-02-20', '2026-03-20', '2026-04-17', '2026-05-15', '2026-06-19', '2026-07-17', '2026-08-21', '2026-09-18', '2026-10-16', '2026-11-20', '2026-12-18', '2027-01-15']


Python Functions

Load in all libraries

In [33]:
#!pip install alpaca-py

# Standard libraries
import os
from datetime import datetime, timedelta

# Environment variables
from dotenv import load_dotenv

# Data handling & plotting
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Alpaca historical stock data
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

# Alpaca historical option data
from alpaca.data.historical.option import OptionHistoricalDataClient
from alpaca.data.requests import OptionLatestQuoteRequest



from datetime import datetime

Connect to Alpaca

In [34]:
load_dotenv()  # Load environment variables from .env file

API_KEY = os.getenv("ALPACA_API_KEY")
SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")

# Create client
client = StockHistoricalDataClient(API_KEY, SECRET_KEY)
option_client = OptionHistoricalDataClient(API_KEY, SECRET_KEY)

# Fetching Data functions

## Stock data

In [16]:
# Calculate date range: last 3 calendar days
end_date = datetime.utcnow()- timedelta(minutes=15)  # Adjust for potential delay
start_date = end_date - timedelta(days=3)

# Request bars within the last 3 days
stock_request = StockBarsRequest(
    symbol_or_symbols="AAPL",
    timeframe=TimeFrame.Minute,
    start=start_date,
    end=end_date,
    limit=100  # maximum bars to fetch
)

bars = client.get_stock_bars(stock_request)

# Convert to DataFrame
df = bars.df.reset_index()

# Sort newest first
df = df.sort_values(by="timestamp", ascending=False)

# Show most recent available bar
print("Most recent available bar:")
print(df.iloc[0])

latest_bar = df.iloc[0]
print("Approximate last bid:", latest_bar['close'])
print("Approximate last ask:", latest_bar['high'])

Most recent available bar:
symbol                              AAPL
timestamp      2026-02-19 10:45:00+00:00
open                              262.79
high                              262.79
low                               262.79
close                             262.79
volume                             144.0
trade_count                          5.0
vwap                              262.79
Name: 99, dtype: object
Approximate last bid: 262.79
Approximate last ask: 262.79


## Option Data

def get_option_quotes(option_symbols):
    # Fetch all quotes in one request
    option_request = OptionLatestQuoteRequest(
        symbol_or_symbols=option_symbols,
        feed="indicative"
    )

    option_quotes = option_client.get_option_latest_quote(option_request)

    quotes_list = []

    for sym, quote_obj in option_quotes.items():
        # Only include if bid/ask exists
        if quote_obj.bid_price is not None and quote_obj.ask_price is not None:
            quotes_list.append({
                "symbol": quote_obj.symbol,
                "bid": quote_obj.bid_price,
                "ask": quote_obj.ask_price,
                "bid_size": quote_obj.bid_size,
                "ask_size": quote_obj.ask_size,
                "timestamp": quote_obj.timestamp
            })

    # Convert to DataFrame
    df_options = pd.DataFrame(quotes_list)

    # Sort by symbol or bid
    df_options = df_options.sort_values(by='symbol').reset_index(drop=True)
    return df_options


In [17]:
def get_option_quotes(option_symbols):
    # Fetch all quotes in one request
    option_request = OptionLatestQuoteRequest(
        symbol_or_symbols=option_symbols,
        feed="indicative"
    )

    option_quotes = option_client.get_option_latest_quote(option_request)

    quotes_list = []

    for sym, quote_obj in option_quotes.items():
        # Only include if bid/ask exists
        if quote_obj.bid_price is not None and quote_obj.ask_price is not None:
            # Extract strike from OPRA symbol
            strike = int(quote_obj.symbol[-8:]) / 1000  # last 8 digits
            quotes_list.append({
                "bid_size": quote_obj.bid_size,
                "bid_price": quote_obj.bid_price,
                "strike": strike,
                "ask_price": quote_obj.ask_price,
                "ask_size": quote_obj.ask_size,
            })

    # Convert to DataFrame
    df_options = pd.DataFrame(quotes_list)

    # Reorder columns
    df_options = df_options[[ "strike", 'bid_size', 'bid_price' ,'ask_price', 'ask_size']]

    # Sort by strike
    df_options = df_options.sort_values(by='strike').reset_index(drop=True)

    return df_options

# Data Processing

Support functions for Df

In [18]:
from datetime import datetime

def make_option_symbol(underlying, expiry_date, strike, option_type="C"):
    """
    Build Alpaca/OPRA option symbol.

    Parameters:
        underlying (str): e.g., "AAPL"
        expiry_date (datetime or str): e.g., "2026-03-20" or datetime object
        strike (float or int): strike price
        option_type (str): "C" or "P"

    Returns:
        str: OPRA-style option symbol
    """
    if isinstance(expiry_date, datetime):
        expiry_str = expiry_date.strftime("%y%m%d")
    else:
        expiry_str = datetime.strptime(expiry_date, "%Y-%m-%d").strftime("%y%m%d")

    # OPRA strike: multiply by 1000, no decimals, pad to 8 digits
    strike_int = int(round(strike * 1000))
    strike_str = f"{strike_int:08d}"

    return f"{underlying}{expiry_str}{option_type}{strike_str}"


def generate_option_symbols_around_price(
    underlying, expiry_date, current_price, option_type="C",
    half=8, step_size=5
):
    """
    Generate a list of OPRA option symbols around a current stock price.

    Parameters:
        underlying (str): e.g., "AAPL"
        expiry_date (str or datetime): e.g., "2026-03-20"
        current_price (float): current stock price
        option_type (str): "C" for calls, "P" for puts
        n_values (int): total number of strikes
        step_size (int): increment between strikes

    Returns:
        list of str: option symbols
    """
    n_values = half * 2 + 1  # Total strikes: half below, half above, plus current
    # Generate strike prices: some below, some above, including current
    strikes = [current_price + (i - half) * step_size for i in range(n_values)]
    # Round to nearest 0.5 or 1 depending on your convention
    strikes = [round(s) for s in strikes]
    
    # Build option symbols
    symbols = [make_option_symbol(underlying, expiry_date, strike, option_type) for strike in strikes]
    return symbols

# Calculation Functions

Options

In [31]:
def get_time_to_expiry(expiry):
    expiry_date = datetime.strptime(expiry, "%Y-%m-%d")
    now = datetime.utcnow()
    return max((expiry_date - now).days / 365.0, 0.0001)
print(get_time_to_expiry("2026-03-20"))

0.0684931506849315


In [ ]:
def vol_surface(K, S, T):

    atm_vol = 0.25
    skew = -0.4 * ((K / S) - 1)
    term = 0.05 * T

    return max(0.05, atm_vol + skew + term)


In [ ]:
import numpy as np
from scipy.stats import norm

def get_time_to_expiry(expiry):
    expiry_date = datetime.strptime(expiry, "%Y-%m-%d")
    now = datetime.utcnow()
    return max((expiry_date - now).days / 365.0, 0.0001)

def bs_call_price(S, K, T, r, sigma):

    if T <= 0:
        return max(0.0, S - K)

    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)


def bs_put_price(S, K, T, r, sigma):

    if T <= 0:
        return max(0.0, K - S)

    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    return K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)


In [ ]:
atm_strike = min(df_calls["strike"], key=lambda K: abs(K - current_price))

atm_call = df_calls[df_calls["strike"] == atm_strike].iloc[0]

mid_price = (atm_call["bid_price"] + atm_call["ask_price"]) / 2

from scipy.optimize import brentq

def implied_vol_call(price, S, K, T, r):

    f = lambda sigma: bs_call(S, K, T, r, sigma) - price

    return brentq(f, 0.001, 5.0)

atm_vol = implied_vol_call(mid_price, current_price, atm_strike, T, r)

def vol_surface(K, S, T, atm_vol):

    m = K / S - 1

    vol = (
        atm_vol
        + skew_slope * m
        + smile_curvature * m**2
        + wing_curvature * abs(m)**3
        - 0.15 * m * np.sqrt(T)
    )

    return np.clip(vol, 0.05, 1.5)

In [ ]:
atm_put = df_puts[df_puts["strike"] == atm_strike].iloc[0]

call_mid = (atm_call["bid_price"] + atm_call["ask_price"]) / 2
put_mid  = (atm_put["bid_price"] + atm_put["ask_price"]) / 2

call_iv = implied_vol_call(call_mid, S, K, T, r)
put_iv  = implied_vol_put(put_mid, S, K, T, r)

atm_vol = (call_iv + put_iv) / 2

In [32]:
from scipy.optimize import brentq
import numpy as np


def implied_vol_call(price, S, K, T, r):

    if price <= 0:
        return 0.20

    def f(sigma):
        return bs_call(S, K, T, r, sigma) - price

    try:
        return brentq(f, 0.001, 5.0)
    except:
        return 0.20


def implied_vol_put(price, S, K, T, r):

    if price <= 0:
        return 0.20

    def f(sigma):
        return bs_put(S, K, T, r, sigma) - price

    try:
        return brentq(f, 0.001, 5.0)
    except:
        return 0.20


def get_option_orderbook_view(
    underlying,
    expiry,
    current_price,
    half=8,
    step_size=5,
):

    r = 0.04
    T = get_time_to_expiry(expiry)

    # Round strike center ONLY
    S_round = round(current_price / step_size) * step_size

    # Generate symbols
    call_symbols = generate_option_symbols_around_price(
        underlying, expiry, S_round, "C", half, step_size
    )

    put_symbols = generate_option_symbols_around_price(
        underlying, expiry, S_round, "P", half, step_size
    )

    df_calls = get_option_quotes(call_symbols)
    df_puts = get_option_quotes(put_symbols)

    # ----------------------------
    # FIND ATM STRIKE
    # ----------------------------

    atm_strike = min(df_calls["strike"], key=lambda K: abs(K - current_price))

    atm_call = df_calls[df_calls["strike"] == atm_strike].iloc[0]
    atm_put = df_puts[df_puts["strike"] == atm_strike].iloc[0]

    call_mid = (atm_call["bid_price"] + atm_call["ask_price"]) / 2
    put_mid = (atm_put["bid_price"] + atm_put["ask_price"]) / 2

    call_iv = implied_vol_call(call_mid, current_price, atm_strike, T, r)
    put_iv = implied_vol_put(put_mid, current_price, atm_strike, T, r)

    atm_vol = (call_iv + put_iv) / 2

    # ----------------------------
    # VOL SURFACE using ATM anchor
    # ----------------------------

    def vol(K):

        m = K / current_price - 1

        vol = (
            atm_vol
            - 0.35 * m
            + 0.60 * m**2
            + 0.25 * abs(m)**3
        )

        return np.clip(vol, 0.05, 1.5)

    # ----------------------------
    # THEO PRICES
    # ----------------------------

    df_calls["call_theo"] = df_calls["strike"].apply(
        lambda K: round(
            bs_call(current_price, K, T, r, vol(K)),
            2,
        )
    )

    df_puts["put_theo"] = df_puts["strike"].apply(
        lambda K: round(
            bs_put(current_price, K, T, r, vol(K)),
            2,
        )
    )

    # Rename columns

    df_calls = df_calls.rename(
        columns={
            "bid_price": "call_bid_price",
            "ask_price": "call_ask_price",
            "bid_size": "call_bid_qty",
            "ask_size": "call_ask_qty",
        }
    )

    df_puts = df_puts.rename(
        columns={
            "bid_price": "put_bid_price",
            "ask_price": "put_ask_price",
            "bid_size": "put_bid_qty",
            "ask_size": "put_ask_qty",
        }
    )

    # Merge

    df_orderbook = pd.merge(df_calls, df_puts, on="strike", how="outer")

    columns_order = [

        "call_bid_qty",
        "call_bid_price",
        "call_theo",
        "call_ask_price",
        "call_ask_qty",

        "strike",

        "put_bid_qty",
        "put_bid_price",
        "put_theo",
        "put_ask_price",
        "put_ask_qty",

    ]

    df_orderbook = (
        df_orderbook[columns_order]
        .sort_values("strike")
        .reset_index(drop=True)
    )

    return df_orderbook

Vol surface


In [ ]:
def vol_surface(K, S, T):

    atm_vol = 0.25

    strike_skew = -0.4 * ((K/S) - 1)

    term_structure = 0.05 * T

    vol = atm_vol + strike_skew + term_structure

    return max(vol, 0.05)

# Complete Orderbook

In [ ]:
def get_option_orderbook_view(
    underlying,
    expiry,
    current_price,
    half=8,
    step_size=5,
):

    r = 0.04
    T = get_time_to_expiry(expiry)

    # ROUND price for strike selection
    S = round(current_price / step_size) * step_size

    # Generate option symbols around price
    call_symbols = generate_option_symbols_around_price(
        underlying, expiry, S, option_type="C", half=half, step_size=step_size
    )
    put_symbols = generate_option_symbols_around_price(
        underlying, expiry, S, option_type="P", half=half, step_size=step_size
    )

    df_calls = get_option_quotes(call_symbols)
    df_puts = get_option_quotes(put_symbols)

    # ---- Compute IV from vol_surface for all strikes ----
    df_calls["call_iv"] = df_calls["strike"].apply(
        lambda K: vol_surface(K, current_price, T)
    )
    df_puts["put_iv"] = df_puts["strike"].apply(
        lambda K: vol_surface(K, current_price, T)
    )

    # ---- Compute theoretical prices ----
    df_calls["call_theo"] = df_calls.apply(
        lambda row: round(
            bs_call(current_price, row["strike"], T, r, row["call_iv"]), 2
        ),
        axis=1,
    )
    df_puts["put_theo"] = df_puts.apply(
        lambda row: round(bs_put(current_price, row["strike"], T, r, row["put_iv"]), 2),
        axis=1,
    )

    # Rename for merging
    df_calls = df_calls.rename(
        columns={
            "bid_price": "call_bid_price",
            "ask_price": "call_ask_price",
            "bid_size": "call_bid_qty",
            "ask_size": "call_ask_qty",
        }
    )
    df_puts = df_puts.rename(
        columns={
            "bid_price": "put_bid_price",
            "ask_price": "put_ask_price",
            "bid_size": "put_bid_qty",
            "ask_size": "put_ask_qty",
        }
    )

    # Merge calls and puts on strike
    df_orderbook = pd.merge(df_calls, df_puts, on="strike", how="outer")

    # Reorder columns, adding IVs at the outer edges
    columns_order = [
        "call_iv",  # left edge
        "call_bid_qty",
        "call_bid_price",
        "call_theo",
        "call_ask_price",
        "call_ask_qty",
        "strike",
        "put_bid_qty",
        "put_bid_price",
        "put_theo",
        "put_ask_price",
        "put_ask_qty",
        "put_iv",  # right edge
    ]

    df_orderbook = (
        df_orderbook.reindex(columns=columns_order)
        .sort_values("strike")
        .reset_index(drop=True)
    )

    return df_orderbook

# Complete files


runner.py

from flask import Flask, render_template, request, jsonify
from flask_socketio import SocketIO
from datetime import datetime, timedelta
import numpy as np
from scipy.stats import norm
import os
from dotenv import load_dotenv
import webbrowser
import threading

from Library_options import get_option_orderbook_view
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

load_dotenv()

API_KEY = os.getenv("ALPACA_API_KEY")
SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")
client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

app = Flask(__name__)
socketio = SocketIO(app, async_mode="threading", cors_allowed_origins="*")

# ---------- global state ----------
state_lock = threading.Lock()
underlying = "MSFT"
expiry = "2026-03-20"


# ---------- Black-Scholes ----------
def bs_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def bs_put(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


# ---------- price ----------
def get_latest_price(symbol):
    try:
        end_date = datetime.utcnow() - timedelta(minutes=15)
        start_date = end_date - timedelta(days=3)
        request_bars = StockBarsRequest(
            symbol_or_symbols=symbol,
            timeframe=TimeFrame.Minute,
            start=start_date,
            end=end_date,
            limit=100,
        )
        bars = client.get_stock_bars(request_bars)
        df = bars.df.reset_index()
        if df.empty:
            return None
        df = df.sort_values("timestamp", ascending=False)
        return float(df.iloc[0]["close"])
    except Exception as e:
        print("Price fetch error:", e)
        return None


# ---------- build dataframe ----------
def build_data(symbol, expiry):
    price = get_latest_price(symbol) or 265
    rounded = 5 * round(price / 5)

    df = get_option_orderbook_view(symbol, expiry, rounded, half=10, step_size=5)

    fill_cols = [
        "call_bid_qty",
        "call_bid_price",
        "call_ask_price",
        "call_ask_qty",
        "put_bid_qty",
        "put_bid_price",
        "put_ask_price",
        "put_ask_qty",
        "call_iv",
        "put_iv",
    ]

    for col in fill_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    df["mid_call"] = (df["call_bid_price"] + df["call_ask_price"]) / 2
    df["mid_put"] = (df["put_bid_price"] + df["put_ask_price"]) / 2
    df["iv"] = df[["call_iv", "put_iv"]].max(axis=1)
    df["iv"] = df["iv"].replace(0, np.nan).fillna(0.2)

    T = max(
        (datetime.strptime(expiry, "%Y-%m-%d") - datetime.utcnow()).days / 365, 1 / 365
    )

    df["call_theo"] = df.apply(
        lambda r: bs_call(price, r["strike"], T, 0.04, r["iv"]), axis=1
    )
    df["put_theo"] = df.apply(
        lambda r: bs_put(price, r["strike"], T, 0.04, r["iv"]), axis=1
    )

    vol_data = {"strike": df["strike"].tolist(), "iv": df["iv"].tolist()}

    return df, price, vol_data


# ---------- websocket loop ----------
def push_loop():
    print("Push loop started")
    while True:
        try:
            with state_lock:
                current_symbol = underlying
                current_expiry = expiry

            df, price, vol = build_data(current_symbol, current_expiry)

            socketio.emit(
                "update",
                {
                    "orderbook": df.to_dict(orient="records"),
                    "price": price,
                    "vol_data": vol,
                    "underlying": current_symbol,
                    "expiry": current_expiry,
                },
            )
        except Exception as e:
            print("Loop error:", e)
        socketio.sleep(1)


# ---------- routes ----------
@app.route("/")
def index():
    return render_template("index.html", underlying=underlying, expiry=expiry)


@app.route("/get_expiries/<symbol>")
def get_expiries(symbol):
    """Return available expiries for a stock (from Library_options)."""
    try:
        price = get_latest_price(symbol) or 265
        rounded = 5 * round(price / 5)
        df = get_option_orderbook_view(
            symbol, expiry=None, strike_price=rounded, half=10, step_size=5
        )
        return jsonify({"expiries": df["expiry"].dropna().unique().tolist()})
    except Exception as e:
        print("Expiry fetch error:", e)
        return jsonify({"expiries": []})


# ---------- socket events ----------
@socketio.on("connect")
def connect():
    print("Client connected")


@socketio.on("change_symbol")
def change_symbol(data):
    global underlying, expiry
    new_symbol = str(data.get("symbol", "")).upper().strip()
    if not new_symbol:
        return
    with state_lock:
        underlying = new_symbol

        # Fetch available expiries
        try:
            df = get_option_orderbook_view(
                new_symbol, expiry=None, half=10, step_size=5
            )
            expiries = df["expiry"].dropna().unique()
            if len(expiries) > 0:
                expiry = sorted(expiries)[0]  # pick earliest expiry
        except Exception as e:
            print("Expiry fetch error:", e)

    print(f"Symbol changed to: {underlying}, expiry set to: {expiry}")


@socketio.on("change_expiry")
def change_expiry(data):
    global expiry
    new_expiry = str(data.get("expiry", "")).strip()
    if not new_expiry:
        return
    with state_lock:
        expiry = new_expiry
    print(f"Expiry changed to: {expiry}")


# ---------- start ----------
if __name__ == "__main__":
    print("Starting server...")
    socketio.start_background_task(push_loop)
    url = "http://127.0.0.1:5050"
    threading.Timer(0.5, lambda: webbrowser.open(url)).start()
    socketio.run(app, host="127.0.0.1", port=5050, debug=False)


index.html

In [ ]:
<!doctype html>
<html>
  <head>
    <title>{{ underlying }} Option Orderbook</title>
    <style>
      body {
        background: #1c1c1c;
        color: white;
        font-family: monospace;
      }
      .controls {
        text-align: center;
        margin-bottom: 10px;
      }
      input,
      select {
        font-size: 14px;
        padding: 4px;
        margin: 0 5px;
      }
      .dashboard {
        display: flex;
        gap: 2cm;
        margin-left: 2cm;
      }
      .table-wrapper {
        position: relative;
      }
      table {
        border-collapse: collapse;
        font-size: 14px;
        background-color: #2c2c2c;
        border: 1px solid #555;
      }
      td,
      th {
        padding: 4px 8px;
        text-align: right;
        background-color: #333;
        border: 1px solid #555;
        transition: background-color 0.5s ease;
      }
      th {
        color: orange;
        background: #444;
      }
      .strike {
        color: yellow;
        font-weight: bold;
        background: #222;
      }
      .green {
        background: #003300 !important;
        color: #00ff90;
      }
      .red {
        background: #330000 !important;
        color: #ff4040;
      }
      .chart-wrapper {
        flex-shrink: 0;
      }
    </style>

    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <script src="https://cdn.socket.io/4.7.2/socket.io.min.js"></script>
  </head>

  <body>
    <div class="controls">
      Stock:
      <input
        list="underlying-list"
        id="underlying-input"
        value="{{ underlying }}"
      />
      <datalist id="underlying-list">
        <option value="AAPL"></option>
        <option value="MSFT"></option>
        <option value="GOOG"></option>
        <option value="AMZN"></option>
        <option value="TSLA"></option>
        <option value="NVDA"></option>
        <option value="META"></option>
      </datalist>

      Expiry:
      <select id="expiry-select">
        <option value="{{ expiry }}">{{ expiry }}</option>
      </select>
    </div>

    <h4 style="text-align: center; color: lightgrey">
      Current Price: <span id="current-price-display">--</span>
    </h4>

    <div class="dashboard">
      <div class="table-wrapper">
        <table id="orderbook">
          <thead>
            <tr>
              <th>Call Bid Qty</th>
              <th>Call Bid</th>
              <th>Call Theo</th>
              <th>Call Ask</th>
              <th>Call Ask Qty</th>
              <th class="strike">Strike</th>
              <th>IV</th>
              <th>Put Bid Qty</th>
              <th>Put Bid</th>
              <th>Put Theo</th>
              <th>Put Ask</th>
              <th>Put Ask Qty</th>
            </tr>
          </thead>
          <tbody></tbody>
        </table>
      </div>

      <div class="chart-wrapper">
        <h4 style="text-align: center; color: lightgrey">Volatility Curve</h4>
        <canvas id="vol-smile" width="500" height="300"></canvas>
      </div>
    </div>

    <script>
      const socket = io({ reconnectionAttempts: 5, reconnectionDelay: 1000 });
      const previousValues = {};
      let volChart = null;
      let previousPrice = null;
      let latestChartPrice = null;

      const symbolInput = document.getElementById("underlying-input");
      const expirySelect = document.getElementById("expiry-select");

      // --- symbol change ---
      function submitSymbolChange() {
        const newSymbol = symbolInput.value.trim().toUpperCase();
        if (!newSymbol) return;
        socket.emit("change_symbol", { symbol: newSymbol });
        fetch(`/get_expiries/${newSymbol}`)
          .then((res) => res.json())
          .then((data) => {
            expirySelect.innerHTML = "";
            data.expiries.forEach((exp) => {
              const option = document.createElement("option");
              option.value = exp;
              option.textContent = exp;
              expirySelect.appendChild(option);
            });
          });
      }

      symbolInput.addEventListener("keydown", (e) => {
        if (e.key === "Enter") submitSymbolChange();
      });
      symbolInput.addEventListener("blur", submitSymbolChange);

      expirySelect.addEventListener("change", () => {
        const newExpiry = expirySelect.value;
        socket.emit("change_expiry", { expiry: newExpiry });
      });

      // --- socket updates ---
      let currentDisplayedSymbol = symbolInput.value; // track what’s in the input

      socket.on("update", (data) => {
        updatePrice(data.price);
        updateTable(data.orderbook);
        updateVolSmile(data.vol_data, data.price);

        // Only update input if the symbol really changed
        if (data.underlying && data.underlying !== currentDisplayedSymbol) {
          symbolInput.addEventListener("input", () => {
            currentDisplayedSymbol = symbolInput.value.trim().toUpperCase();
          });
        }
      });

      function updatePrice(price) {
        const priceElem = document.getElementById("current-price-display");
        const newPrice = price.toFixed(2);
        if (previousPrice !== null && newPrice != previousPrice) {
          priceElem.classList.add(newPrice > previousPrice ? "green" : "red");
          setTimeout(() => priceElem.classList.remove("green", "red"), 400);
        }
        priceElem.textContent = newPrice;
        previousPrice = newPrice;
      }

      function updateTable(data) {
        const tbody = document.querySelector("#orderbook tbody");
        tbody.innerHTML = "";
        data
          .slice()
          .reverse()
          .forEach((row) => {
            const tr = document.createElement("tr");
            function buildCell(value, key, decimals = 2) {
              const td = document.createElement("td");
              if (value !== null && value !== undefined) {
                const formatted =
                  typeof value === "number" ? value.toFixed(decimals) : value;
                td.textContent = formatted;
                const prev = previousValues[key];
                if (prev !== undefined) {
                  if (value > prev) td.classList.add("green");
                  if (value < prev) td.classList.add("red");
                  setTimeout(() => td.classList.remove("green", "red"), 400);
                }
                previousValues[key] = value;
              }
              return td;
            }

            const strike = row.strike;
            tr.appendChild(buildCell(row.call_bid_qty, "cbq_" + strike, 0));
            tr.appendChild(buildCell(row.call_bid_price, "cbp_" + strike));
            tr.appendChild(buildCell(row.call_theo, "ct_" + strike));
            tr.appendChild(buildCell(row.call_ask_price, "cap_" + strike));
            tr.appendChild(buildCell(row.call_ask_qty, "caq_" + strike, 0));

            const strikeCell = document.createElement("td");
            strikeCell.textContent = row.strike?.toFixed(2) ?? "";
            strikeCell.className = "strike";
            tr.appendChild(strikeCell);

            const ivCell = document.createElement("td");
            ivCell.textContent = row.iv ? (row.iv * 100).toFixed(2) + "%" : "";
            tr.appendChild(ivCell);

            tr.appendChild(buildCell(row.put_bid_qty, "pbq_" + strike, 0));
            tr.appendChild(buildCell(row.put_bid_price, "pbp_" + strike));
            tr.appendChild(buildCell(row.put_theo, "pt_" + strike));
            tr.appendChild(buildCell(row.put_ask_price, "pap_" + strike));
            tr.appendChild(buildCell(row.put_ask_qty, "paq_" + strike, 0));

            tbody.appendChild(tr);
          });
      }

      function updateVolSmile(volData, currentPrice) {
        const ctx = document.getElementById("vol-smile").getContext("2d");
        const ivData = volData.strike.map((s, i) => ({
          x: Number(s),
          y: Number(volData.iv[i]) * 100,
        }));
        latestChartPrice = currentPrice;

        const priceLinePlugin = {
          id: "priceLine",
          afterDraw(chart) {
            if (latestChartPrice === null) return;
            const xScale = chart.scales.x,
              yScale = chart.scales.y;
            const xPos = xScale.getPixelForValue(latestChartPrice);
            const ctx = chart.ctx;
            ctx.save();
            ctx.beginPath();
            ctx.moveTo(xPos, yScale.top);
            ctx.lineTo(xPos, yScale.bottom);
            ctx.strokeStyle = "red";
            ctx.lineWidth = 2;
            ctx.stroke();
            ctx.restore();
          },
        };

        if (!volChart) {
          volChart = new Chart(ctx, {
            type: "line",
            data: {
              datasets: [
                {
                  label: "IV",
                  data: ivData,
                  borderColor: "cyan",
                  pointRadius: 3,
                  fill: false,
                },
              ],
            },
            options: {
              scales: {
                x: { type: "linear", title: { display: true, text: "Strike" } },
                y: { title: { display: true, text: "IV %" } },
              },
            },
            plugins: [priceLinePlugin],
          });
        } else {
          volChart.data.datasets[0].data = ivData;
          volChart.update();
        }
      }
    </script>
  </body>
</html>


SyntaxError: invalid decimal literal (3476902827.py, line 10)

Library

# pip install alpaca-py

# Standard libraries
import os
from datetime import datetime, timedelta

from dotenv import load_dotenv

# Data handling & plotting
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from scipy.stats import norm

# Alpaca historical stock data
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

# Alpaca historical option data
from alpaca.data.historical.option import OptionHistoricalDataClient
from alpaca.data.requests import OptionLatestQuoteRequest

load_dotenv()  # Load environment variables from .env file

API_KEY = os.getenv("ALPACA_API_KEY")
SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")

# ALPACA_API_KEY = "PKQBAIAQMHCIAFQMQUVD457BK3"
# ALPACA_SECRET_KEY = "BME8QF8GP6GmWkdJXoCDYkNQHP2fiRSq7yQA71KbqvjS"

# Create client
client = StockHistoricalDataClient(API_KEY, SECRET_KEY)
option_client = OptionHistoricalDataClient(API_KEY, SECRET_KEY)

# stock_bars_request

# options_bars_request


def get_option_quotes(option_symbols):
    # Fetch all quotes in one request
    option_request = OptionLatestQuoteRequest(
        symbol_or_symbols=option_symbols, feed="indicative"
    )

    option_quotes = option_client.get_option_latest_quote(option_request)

    quotes_list = []

    for sym, quote_obj in option_quotes.items():
        # Only include if bid/ask exists
        if quote_obj.bid_price is not None and quote_obj.ask_price is not None:
            # Extract strike from OPRA symbol
            strike = int(quote_obj.symbol[-8:]) / 1000  # last 8 digits
            quotes_list.append(
                {
                    "bid_size": quote_obj.bid_size,
                    "bid_price": quote_obj.bid_price,
                    "strike": strike,
                    "ask_price": quote_obj.ask_price,
                    "ask_size": quote_obj.ask_size,
                }
            )

    # Convert to DataFrame
    df_options = pd.DataFrame(quotes_list)

    # Reorder columns
    df_options = df_options[
        ["strike", "bid_size", "bid_price", "ask_price", "ask_size"]
    ]

    # Sort by strike
    df_options = df_options.sort_values(by="strike").reset_index(drop=True)

    return df_options


from datetime import datetime


def make_option_symbol(underlying, expiry_date, strike, option_type="C"):
    """
    Build Alpaca/OPRA option symbol.

    Parameters:
        underlying (str): e.g., "AAPL"
        expiry_date (datetime or str): e.g., "2026-03-20" or datetime object
        strike (float or int): strike price
        option_type (str): "C" or "P"

    Returns:
        str: OPRA-style option symbol
    """
    if isinstance(expiry_date, datetime):
        expiry_str = expiry_date.strftime("%y%m%d")
    else:
        expiry_str = datetime.strptime(expiry_date, "%Y-%m-%d").strftime("%y%m%d")

    # OPRA strike: multiply by 1000, no decimals, pad to 8 digits
    strike_int = int(round(strike * 1000))
    strike_str = f"{strike_int:08d}"

    return f"{underlying}{expiry_str}{option_type}{strike_str}"


# to find market price and generate orderbook around it


def generate_option_symbols_around_price(
    underlying, expiry_date, current_price, option_type="C", half=8, step_size=5
):
    """
    Generate a list of OPRA option symbols around a current stock price.

    Parameters:
        underlying (str): e.g., "AAPL"
        expiry_date (str or datetime): e.g., "2026-03-20"
        current_price (float): current stock price
        option_type (str): "C" for calls, "P" for puts
        n_values (int): total number of strikes
        step_size (int): increment between strikes

    Returns:
        list of str: option symbols
    """
    n_values = half * 2 + 1  # Total strikes: half below, half above, plus current
    # Generate strike prices: some below, some above, including current
    strikes = [current_price + (i - half) * step_size for i in range(n_values)]
    # Round to nearest 0.5 or 1 depending on your convention
    strikes = [round(s) for s in strikes]

    # Build option symbols
    symbols = [
        make_option_symbol(underlying, expiry_date, strike, option_type)
        for strike in strikes
    ]
    return symbols


def get_time_to_expiry(expiry):
    expiry_date = datetime.strptime(expiry, "%Y-%m-%d")
    now = datetime.utcnow()
    return max((expiry_date - now).days / 365.0, 0.0001)


import numpy as np


def svi_vol_surface(
    K, S, T, atm_vol=0.22, skew=-0.4, curvature=0.6, wing=0.2, smile_shift=0.0, r=0.04
):
    """
    SVI volatility surface

    Parameters
    ----------
    K : strike
    S : spot
    T : time to expiry (years)

    atm_vol : ATM implied vol
    skew : skew slope (negative for equities)
    curvature : smile curvature
    wing : wing steepness
    smile_shift : horizontal shift

    Returns
    -------
    implied volatility
    """

    if T <= 0:
        return atm_vol

    # forward approx (can add rates/div later)
    F = S * np.exp(r * T)  # forward price, ignoring dividends for simplicity

    # log-moneyness
    k = np.log(K / F)

    # convert atm vol to total variance
    w_atm = atm_vol**2 * T

    # SVI parameters derived from intuitive inputs
    a = w_atm * 0.5
    b = wing * w_atm
    rho = skew
    m = smile_shift
    sigma = curvature

    total_variance = a + b * (rho * (k - m) + np.sqrt((k - m) ** 2 + sigma**2))

    vol = np.sqrt(total_variance / T)

    return np.clip(vol, 0.01, 3.0)


# Black-Scholes formulas for call and put theoretical prices
def bs_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def bs_put(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


# -- final option orderbook view function --


def get_option_orderbook_view(
    underlying,
    expiry,
    current_price,
    half=8,
    step_size=5,
):

    r = 0.04
    T = get_time_to_expiry(expiry)

    # ROUND price for strike selection
    S = round(current_price / step_size) * step_size

    # Generate option symbols around price
    call_symbols = generate_option_symbols_around_price(
        underlying, expiry, S, option_type="C", half=half, step_size=step_size
    )
    put_symbols = generate_option_symbols_around_price(
        underlying, expiry, S, option_type="P", half=half, step_size=step_size
    )

    df_calls = get_option_quotes(call_symbols)
    df_puts = get_option_quotes(put_symbols)

    # ---- Compute IV from vol_surface for all strikes ----
    df_calls["call_iv"] = df_calls["strike"].apply(
        lambda K: svi_vol_surface(K, current_price, T, r)
    )
    df_puts["put_iv"] = df_puts["strike"].apply(
        lambda K: svi_vol_surface(K, current_price, T, r)
    )

    # ---- Compute theoretical prices ----
    df_calls["call_theo"] = df_calls.apply(
        lambda row: round(
            bs_call(current_price, row["strike"], T, r, row["call_iv"]), 2
        ),
        axis=1,
    )
    df_puts["put_theo"] = df_puts.apply(
        lambda row: round(bs_put(current_price, row["strike"], T, r, row["put_iv"]), 2),
        axis=1,
    )

    # Rename for merging
    df_calls = df_calls.rename(
        columns={
            "bid_price": "call_bid_price",
            "ask_price": "call_ask_price",
            "bid_size": "call_bid_qty",
            "ask_size": "call_ask_qty",
        }
    )
    df_puts = df_puts.rename(
        columns={
            "bid_price": "put_bid_price",
            "ask_price": "put_ask_price",
            "bid_size": "put_bid_qty",
            "ask_size": "put_ask_qty",
        }
    )

    # Merge calls and puts on strike
    df_orderbook = pd.merge(df_calls, df_puts, on="strike", how="outer")

    # Reorder columns, adding IVs at the outer edges
    columns_order = [
        "call_iv",  # left edge
        "call_bid_qty",
        "call_bid_price",
        "call_theo",
        "call_ask_price",
        "call_ask_qty",
        "strike",
        "put_bid_qty",
        "put_bid_price",
        "put_theo",
        "put_ask_price",
        "put_ask_qty",
        "put_iv",  # right edge
    ]

    df_orderbook = (
        df_orderbook.reindex(columns=columns_order)
        .sort_values("strike")
        .reset_index(drop=True)
    )

    return df_orderbook
